## Natural Language Processing Spring 2026
---
# End-to-End Pipeline: Pre-training, SFT, Checkpointing, and Evaluation

**Objective:** Build a robust data pipeline and two-phase training loop for our MiniMind Causal Language Model, save the resulting models, and evaluate them.

To ensure this notebook runs completely independently:
* **Step 1** downloads the required datasets from Hugging Face.
* **Step 2** explores the downloaded datasets.
* **Step 3** includes the full implementation of the `MiniMind` architecture.
* **Step 4** constructs both the `PretrainDataset` and the `SFTDataset`.
* **Step 5** implements a memory-efficient training loop featuring Gradient Accumulation, Clipping, and **Checkpoint Saving**.
* **Step 6** executes Phase 1 (Pre-training) and Phase 2 (SFT), saving checkpoints for both.
* **Step 7** introduces a generation function and performs a **Side-by-Side Computation** comparing the Pre-trained model against the SFT model.

## Step 0: Setup and Imports

In [ ]:
!pip install -q huggingface_hub datasets

import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
import math
import time
from contextlib import nullcontext
from datasets import load_dataset, Features, Value
from huggingface_hub import hf_hub_download

## Step 1: Download Datasets from Hugging Face

In [ ]:
# Define the Hugging Face repository and the files we want to download
repo_id = "jingyaogong/minimind_dataset"
files_to_download = [
    "pretrain_t2t_mini.jsonl",
    "sft_t2t_mini.jsonl"
]

# Create the target directory as expected by the pipeline
dataset_dir = "./dataset"
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading datasets from Hugging Face...")
for file_name in files_to_download:
    print(f"Fetching {file_name}...")
    hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
        repo_type="dataset",
        local_dir=dataset_dir
    )

print("\nAll datasets successfully downloaded to the './dataset' directory!")

## Step 2: Dataset Exploration and Statistics

In [ ]:
dataset_files = [
    "./dataset/pretrain_t2t_mini.jsonl",
    "./dataset/sft_t2t_mini.jsonl"
]

print("--- Dataset Statistics & Examples ---\n")

for file_path in dataset_files:
    if not os.path.exists(file_path):
        continue

    count = 0
    first_sample = None
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i == 0: first_sample = json.loads(line)
            count += 1

    print(f"📌 Dataset: {os.path.basename(file_path)}")
    print(f"Total Samples: {count:,}")
    print("Example Sample:")
    print(json.dumps(first_sample, ensure_ascii=False, indent=2))
    print("-" * 50 + "\n")

## Step 3: MiniMind Architecture (Standalone)

In [ ]:
# --- 1. Configuration & Norm ---
class MiniMindConfig:
    def __init__(self):
        self.vocab_size = 6400
        self.hidden_size = 256  # Reduced for faster demonstration
        self.num_hidden_layers = 2
        self.num_attention_heads = 8
        self.num_key_value_heads = 4 
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.intermediate_size = 1024
        self.max_position_embeddings = 2048
        self.rms_norm_eps = 1e-6
        self.dropout = 0.0
        self.tie_word_embeddings = True

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * normed.float()).type_as(x)

# --- 2. Rotary Position Embeddings (RoPE) ---
def precompute_freqs_cis(dim: int, end: int, rope_base: float = 1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        half_dim = x.shape[-1] // 2
        return torch.cat((-x[..., half_dim:], x[..., :half_dim]), dim=-1)
    cos_unsqueezed = cos.unsqueeze(0).unsqueeze(2)
    sin_unsqueezed = sin.unsqueeze(0).unsqueeze(2)
    q_embed = (q * cos_unsqueezed) + (rotate_half(q) * sin_unsqueezed)
    k_embed = (k * cos_unsqueezed) + (rotate_half(k) * sin_unsqueezed)
    return q_embed, k_embed

# --- 3. Attention & FeedForward ---
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    x_expanded = x[:, :, :, None, :].expand(bs, slen, num_kv_heads, n_rep, head_dim)
    return x_expanded.reshape(bs, slen, num_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, self.n_local_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_local_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, x, position_embeddings):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        xq = xq.transpose(1, 2)
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)
        scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
        scores = scores + mask
        attention_output = F.softmax(scores, dim=-1) @ xv
        attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)
        return self.o_proj(attention_output)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        gated_features = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(gated_features)

# --- 4. Transformer Block & LM Wrapper ---
class MiniMindBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config)
    def forward(self, x, position_embeddings):
        x = x + self.self_attn(self.input_layernorm(x), position_embeddings)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([MiniMindBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.embed_tokens.weight = self.lm_head.weight
            
    def forward(self, input_ids, position_embeddings, labels=None):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x, position_embeddings)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100 
            )
        return logits, loss

## Step 4: The Data Pipelines (Pre-training and SFT)

In [ ]:
# --- 1. Pretrain Dataset ---
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100 # Mask padding
        return input_ids, labels

# --- 2. SFT Dataset ---
class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        features = Features({'conversations': [{'role': Value('string'), 'content': Value('string'), 'reasoning_content': Value('string'), 'tools': Value('string'), 'tool_calls': Value('string')}]})
        self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
        self.bos_id = [tokenizer.bos_token_id]
        self.eos_id = [tokenizer.eos_token_id]

    def __len__(self):
        return len(self.samples)

    def generate_labels(self, input_ids):
        """Masks out user prompts so only the assistant's response is learned."""
        labels = [-100] * len(input_ids)
        # For this standalone notebook, we simulate masking by only predicting the second half of the sequence.
        start_assistant = len(input_ids) // 2
        for j in range(start_assistant, len(input_ids)):
            if input_ids[j] != self.tokenizer.pad_token_id:
                labels[j] = input_ids[j]       
        return labels

    def __getitem__(self, index):
        sample = self.samples[index]
        prompt = str(sample['conversations'])
        input_ids = self.tokenizer(prompt, add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))
        labels = self.generate_labels(input_ids)
        
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

## Step 5: The Core Training Loop (with Checkpointing)

We have added a new `save_path` parameter to `train_epoch_demo`. At the end of each training phase, the model will dump its `state_dict` to disk, allowing us to load these distinct checkpoints later for evaluation.

In [ ]:
def train_epoch_demo(model, loader, optimizer, scaler, args, pos_embs, phase_name, save_path):
    device_type = "cuda" if "cuda" in args.device else "cpu"
    dtype = torch.float16
    autocast_ctx = nullcontext() if device_type == "cpu" else torch.cuda.amp.autocast(dtype=dtype)
    
    model.train()
    print(f"\n>>> Starting Phase: {phase_name} <<<")
    for step, (input_ids, labels) in enumerate(loader, start=1):
        input_ids = input_ids.to(args.device)
        labels = labels.to(args.device)

        with autocast_ctx:
            logits, loss = model(input_ids, position_embeddings=pos_embs, labels=labels)
            loss = loss / args.accumulation_steps

        scaler.scale(loss).backward()

        if step % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % args.log_interval == 0:
            print(f"[{phase_name}] Step {step}: Loss = {loss.item() * args.accumulation_steps:.4f}")
            
    if step % args.accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    # Save Model Checkpoint
    print(f"Saving {phase_name} checkpoint to {save_path}...")
    torch.save(model.state_dict(), save_path)
    print("Checkpoint saved successfully!")

## Step 6: End-to-End Execution (Phase 1 & Phase 2)
We will now simulate the entire LLM training lifecycle using the `_mini` subsets. We'll save `pretrain_model.pth` and `sft_model.pth` to disk.

In [ ]:
class TrainArgs:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    epochs = 1
    learning_rate = 5e-4
    accumulation_steps = 2
    grad_clip = 1.0
    log_interval = 5
    max_seq_len = 32

args = TrainArgs()
config = MiniMindConfig()
os.makedirs("./checkpoints", exist_ok=True)

model = MiniMindForCausalLM(config).to(args.device)
pos_embs = precompute_freqs_cis(config.head_dim, args.max_seq_len)
pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))

class MockTokenizer:
    bos_token_id = 1
    eos_token_id = 2
    pad_token_id = 0
    def __call__(self, text, add_special_tokens=False, max_length=512, truncation=True):
        class Output:
            input_ids = [(ord(c) % 6300) + 10 for c in text][:max_length]
        return Output()
    def decode(self, token_ids):
        return "".join([chr(t - 10) if 32 <= (t-10) <= 126 else "*" for t in token_ids])

tokenizer = MockTokenizer()
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
scaler = torch.cuda.amp.GradScaler(enabled=("cuda" in args.device))

print("=== MiniMind End-to-End Training Simulation ===")

# --- PHASE 1: PRE-TRAINING ---
pretrain_path = "./dataset/pretrain_t2t_mini.jsonl"
if os.path.exists(pretrain_path):
    ds_pretrain = PretrainDataset(pretrain_path, tokenizer, max_length=args.max_seq_len)
    ds_pretrain.samples = ds_pretrain.samples.select(range(20)) 
    loader_pretrain = DataLoader(ds_pretrain, batch_size=4, shuffle=True)
    
    train_epoch_demo(model, loader_pretrain, optimizer, scaler, args, pos_embs, 
                     phase_name="Pre-training", 
                     save_path="./checkpoints/pretrain_model.pth")

# --- PHASE 2: SUPERVISED FINE-TUNING ---
sft_path = "./dataset/sft_t2t_mini.jsonl"
if os.path.exists(sft_path):
    ds_sft = SFTDataset(sft_path, tokenizer, max_length=args.max_seq_len)
    ds_sft.samples = ds_sft.samples.select(range(20))
    loader_sft = DataLoader(ds_sft, batch_size=4, shuffle=True)
    
    # Lower learning rate for fine-tuning phase
    for param_group in optimizer.param_groups:
        param_group['lr'] = 5e-5
        
    train_epoch_demo(model, loader_sft, optimizer, scaler, args, pos_embs, 
                     phase_name="Supervised Fine-Tuning", 
                     save_path="./checkpoints/sft_model.pth")


## Step 7: Side-by-Side Computation & Evaluation
With our checkpoints saved to disk, we can now initialize two separate models in memory—one loaded with the Pre-trained weights, and the other with the SFT weights.

We'll build a basic autoregressive text generation function and pass the same prompt to both models. Since they have been trained on different data objectives (word chain completion vs. instruction following), we can compute and compare their responses side-by-side.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=15):
    model.eval()
    input_ids = torch.tensor([tokenizer(prompt, max_length=args.max_seq_len).input_ids], dtype=torch.long).to(args.device)
    
    # Precompute RoPE for the max possible length during generation
    pos_embs = precompute_freqs_cis(config.head_dim, input_ids.size(1) + max_new_tokens)
    pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            current_seq_len = input_ids.size(1)
            current_pos_embs = (pos_embs[0][:current_seq_len], pos_embs[1][:current_seq_len])
            
            logits, _ = model(input_ids, position_embeddings=current_pos_embs)
            # Greedy decoding: pick the token with the highest probability
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
    return tokenizer.decode(input_ids[0].cpu().tolist())

print("\n=== Loading Models for Side-by-Side Evaluation ===")
# 1. Initialize Base Architecture for Both
model_pretrain = MiniMindForCausalLM(config).to(args.device)
model_sft = MiniMindForCausalLM(config).to(args.device)

# 2. Load the Checkpoints
if os.path.exists("./checkpoints/pretrain_model.pth") and os.path.exists("./checkpoints/sft_model.pth"):
    model_pretrain.load_state_dict(torch.load("./checkpoints/pretrain_model.pth"))
    model_sft.load_state_dict(torch.load("./checkpoints/sft_model.pth"))
    print("Checkpoints successfully loaded into memory.")
    
    # 3. Test with a sample prompt
    test_prompt = "Hello Assistant, can you help me?"
    print(f"\n[Prompt]: {test_prompt}\n")
    
    # Note: Because this notebook uses a severely truncated 20-sample dataset, 
    # a micro 256-dim model, and a mock ASCII tokenizer, the output will look like 
    # random characters. In a full run, this is where you observe the stark 
    # behavioral difference between completion (Pretrain) and chatting (SFT).
    
    out_pretrain = generate_text(model_pretrain, tokenizer, test_prompt, max_new_tokens=10)
    out_sft = generate_text(model_sft, tokenizer, test_prompt, max_new_tokens=10)
    
    print("=== SIDE-BY-SIDE COMPUTATION ===")
    print("🤖 Pre-trained Model Output:")
    print(f"> {out_pretrain}\n")
    print("🤖 SFT Model Output:")
    print(f"> {out_sft}\n")
    print("==================================")
else:
    print("Checkpoints not found. Ensure Step 6 completed successfully.")